# 00 - Data Pipeline

Section 1: data handling and memory management.

This notebook checks that the raw files cover the Dec 16-22 forecast week, compares memory usage between a naive load and `DataLoader`'s two-pass chunked aggregation, builds the processed dataset from all 62 raw days, and ranks the 10,000 grid squares by total traffic to pick the three squares used for the rest of the project.

All the loading logic lives in `forecasting/data.py`; this notebook just calls it and reports what came back.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import time

import pandas as pd

from forecasting.data import DataLoader, naive_load_day, measure_peak_memory

RAW_DIR = ROOT / "data" / "raw"
DAILY_DIR = ROOT / "data" / "processed" / "daily"
COMBINED_PATH = ROOT / "data" / "processed" / "internet_traffic.parquet"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [2]:
raw_files = sorted(RAW_DIR.glob("sms-call-internet-mi-*.txt"))
dates = [f.stem.replace("sms-call-internet-mi-", "") for f in raw_files]
print(f"{len(raw_files)} raw files found, {dates[0]} .. {dates[-1]}")

required_week = {f"2013-12-{d:02d}" for d in range(16, 23)}
missing = required_week - set(dates)
if missing:
    raise RuntimeError(f"Missing raw files for the Dec 16-22 evaluation week: {sorted(missing)}")
print("Dec 16-22 evaluation week: all 7 daily files present.")

62 raw files found, 2013-11-01 .. 2014-01-01
Dec 16-22 evaluation week: all 7 daily files present.


All 62 daily files are present, spanning 2013-11-01 through 2014-01-01, which covers the Dec 16-22 week needed for the forecasting evaluation in `experiments.ipynb`. Nothing to download or patch before continuing.

In [3]:
# Naive read_csv vs. DataLoader's two-pass approach, each in an isolated subprocess.
sample_file = raw_files[0]
tmp_out = RESULTS_DIR / "_memory_demo.parquet"

naive_peak = measure_peak_memory(naive_load_day, sample_file)
optimized_peak = measure_peak_memory(DataLoader().process_day, sample_file, tmp_out)
tmp_out.unlink(missing_ok=True)

memory_table = pd.DataFrame([
    {"approach": "Naive (read_csv, all columns, default dtypes)", "peak_MB": naive_peak / 1e6},
    {"approach": "DataLoader (two-pass, chunked, downcast, preallocated array)", "peak_MB": optimized_peak / 1e6},
])
memory_table

,approach,peak_MB
0,"Naive (read_csv, all columns, default dtypes)",453.984256
1,"DataLoader (two-pass, chunked, downcast, preal...",206.364672


The naive approach loads all 8 raw columns at default 64-bit dtypes in one `read_csv` call, so its peak memory scales directly with file size (~322MB raw -> several hundred MB resident) - running all 62 files this way at once wouldn't fit comfortably on a laptop-class machine. `DataLoader.process_day` bounds memory two ways instead: it only holds one chunk of raw rows at a time, and rather than concatenating per-chunk DataFrames it aggregates straight into a preallocated `(10,000 squares x ~144 timestamps)` float32 array - a few MB, fixed regardless of file size or chunk count. The trade-off is the extra bookkeeping of the two-pass timestamp scan, in exchange for a memory bound that no longer depends on raw file size.

In [4]:
# Build the full processed dataset (idempotent - skips already-processed days).
loader = DataLoader()

t0 = time.time()
stats = loader.build_all(RAW_DIR, DAILY_DIR)
build_seconds = time.time() - t0
print(f"Processed {len(stats)} new day(s) in {build_seconds:.1f}s "
      f"({len(list(DAILY_DIR.glob('*.parquet')))} day-files total on disk).")

t0 = time.time()
combined = loader.combine(DAILY_DIR, COMBINED_PATH)
print(f"Combined into {COMBINED_PATH.name}: {combined.shape} in {time.time()-t0:.1f}s")
combined.head()

Processed 0 new day(s) in 0.0s (62 day-files total on disk).


Combined into internet_traffic.parquet: (89280000, 3) in 64.3s


,square_id,internet_traffic,timestamp
0,1,11.028366,2013-11-01 00:00:00
1,1,11.127101,2013-11-01 00:10:00
2,1,10.892771,2013-11-01 00:20:00
3,1,8.622424,2013-11-01 00:30:00
4,1,8.009928,2013-11-01 00:40:00


Each daily file is processed independently and idempotently (re-running this cell only processes days without existing output), then `combine()` concatenates the 62 per-day Parquet files into one dataset sorted by `(square_id, timestamp)`. The result is a compact, columnar int16/float32 file - down from ~20GB of raw tab-separated text - small enough to load in full for the EDA notebook, and to filter efficiently per square (via `SquareSeries`'s Parquet predicate pushdown) in the modeling notebooks.

In [5]:
totals = combined.groupby("square_id")["internet_traffic"].sum().sort_values(ascending=False)
top3 = totals.head(3)
print("Top 3 squares by total internet traffic (Nov 1 - Jan 1):")
print(top3)

top_squares_info = {
    "top3_square_ids": [int(s) for s in top3.index],
    "top3_totals": {int(s): float(v) for s, v in top3.items()},
    "fixed_reference_squares": [4159, 4556],
}
with open(RESULTS_DIR / "top_squares.json", "w") as f:
    json.dump(top_squares_info, f, indent=2)
top_squares_info

Top 3 squares by total internet traffic (Nov 1 - Jan 1):
square_id
5161    12740060.0
5059    11170854.0
5259    10485779.0
Name: internet_traffic, dtype: float32


{'top3_square_ids': [5161, 5059, 5259],
 'top3_totals': {5161: 12740060.0, 5059: 11170854.0, 5259: 10485779.0},
 'fixed_reference_squares': [4159, 4556]}

These three squares - the highest-traffic areas over the full two-month window - are used for every later analysis and forecasting experiment. The result is saved to `results/top_squares.json` so downstream notebooks read it back instead of recomputing it, keeping "which squares we're forecasting" defined in one place.